# Demo — end-to-end inference
One clip, start to finish: audio -> segment graph -> caption -> fused prediction.
Requires `results/task3_fusion.pt` (run `python train.py --task 3` first) and
a cached graph in `data/processed/task2_graphs/`.

In [ ]:
import sys, json
sys.path.insert(0, '..')
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer
from torch_geometric.data import Data, Batch

from src.fusion_model import FusionTagger

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('../data/processed')

mtat = pd.read_csv(DATA_DIR / 'mtat_master.csv')
meta = json.load(open(DATA_DIR / 'dataset_meta.json'))
TOP50 = meta['top50_tags']
graph_meta = pd.read_csv(DATA_DIR / 'task2_graphs' / 'graph_metadata.csv')
print(f'{len(graph_meta)} cached graphs available')

## Pick one test clip and run it through the full pipeline

In [ ]:
row = graph_meta[graph_meta.split == 'test'].iloc[0]
clip_row = mtat[mtat.clip_id == row.clip_id].iloc[0]

print('clip_id :', row.clip_id)
print('caption :', clip_row.caption)
print('true tags:', [t for t in TOP50 if clip_row[t] == 1])

# 1. Load the cached graph (built from audio in Task 2)
g = torch.load(row.graph_path, map_location='cpu', weights_only=False)
print('\ngraph: ', g['x'].shape[0], 'nodes,', g['edge_index'].shape[1], 'edges')

# 2. Tokenize the caption (Task 1's text side)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
enc = tokenizer(str(clip_row.caption), truncation=True, padding='max_length',
                max_length=128, return_tensors='pt')

data = Data(x=g['x'].float(), edge_index=g['edge_index'].long())
data.input_ids = enc['input_ids']
data.attention_mask = enc['attention_mask']
batch = Batch.from_data_list([data]).to(DEVICE)

## 3. Load the trained fusion model and predict

In [ ]:
ckpt = torch.load('../results/task3_fusion.pt', map_location=DEVICE, weights_only=False)
node_dim = g['x'].shape[1]

model = FusionTagger(node_dim, len(TOP50), mode='cross_attn').to(DEVICE)
model.load_state_dict(ckpt)
model.eval()

with torch.no_grad():
    logits, z = model(batch)
    probs = torch.sigmoid(logits)[0].cpu().numpy()

import numpy as np
top5 = np.argsort(-probs)[:5]
print('Predicted tags:')
for k in top5:
    print(f'  {TOP50[k]:15s} {probs[k]:.3f}')